# 🚢 Titanic Survival Classification
## Notebook 3 — Feature Engineering

### Objective

Transform the original Titanic passenger data into meaningful
machine-learning features.

Feature engineering is used to extract additional information from
the raw variables and make the data more informative for classification.

### Features Created

- FamilySize
- IsAlone
- AgeGroup
- Title
- Deck
- CabinKnown
- TicketGroupSize
- FarePerPerson
- TicketPrefix
- SmallFamily
- LargeFamily
- IsMother

The engineered dataset will be saved inside:

`data/processed/`

In [ ]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import sys
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

# Add project root to Python path
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:")
print(PROJECT_ROOT)

In [ ]:
# ============================================================
# LOAD DATA
# ============================================================

TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "train.csv"

df = pd.read_csv(TRAIN_PATH)

print(f"Dataset shape: {df.shape}")

display(df.head())


## 1. Understand the Original Features

Before creating new variables, inspect the original columns.

Important variables include:

- `Pclass`: passenger class and socio-economic proxy
- `Sex`: passenger sex
- `Age`: passenger age
- `SibSp`: siblings/spouses aboard
- `Parch`: parents/children aboard
- `Fare`: passenger fare
- `Cabin`: cabin information
- `Embarked`: embarkation port
- `Name`: passenger name
- `Ticket`: ticket identifier


In [ ]:
# ============================================================
# ORIGINAL COLUMNS
# ============================================================

print("Original columns:\n")

for i, column in enumerate(df.columns, start=1):
    print(f"{i:02d}. {column}")

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))


## 2. Create Family Size

Family size is calculated as:

FamilySize = SibSp + Parch + 1

The additional 1 represents the passenger themselves.

This gives a single variable representing the size of the
passenger's immediate family group aboard the Titanic.


In [ ]:
# ============================================================
# FAMILY SIZE
# ============================================================

df["FamilySize"] = (
    df["SibSp"] +
    df["Parch"] +
    1
)

display(
    df[
        [
            "SibSp",
            "Parch",
            "FamilySize"
        ]
    ].head(10)
)


## 3. Create IsAlone

Passengers with a family size of one are considered to be
traveling alone.

This converts family size into a simple binary indicator.


In [ ]:
# ============================================================
# IS ALONE
# ============================================================

df["IsAlone"] = (
    df["FamilySize"] == 1
).astype(int)

print(
    df["IsAlone"]
    .value_counts()
    .rename({
        0: "Not Alone",
        1: "Alone"
    })
)


## 4. Create Age Groups

Age is converted into meaningful age categories.

The categories are:

- Infant
- Child
- Teen
- Young Adult
- Adult
- Middle Age
- Senior

Categorizing age allows us to investigate survival patterns
across different life stages.


In [ ]:
# ============================================================
# AGE GROUP
# ============================================================

age_bins = [
    -np.inf,
    5,
    12,
    18,
    30,
    45,
    60,
    np.inf
]

age_labels = [
    "Infant",
    "Child",
    "Teen",
    "Young Adult",
    "Adult",
    "Middle Age",
    "Senior"
]

df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=age_bins,
    labels=age_labels
)

display(
    df[
        ["Age", "AgeGroup"]
    ].head(15)
)


## 5. Extract Passenger Titles

Passenger names contain titles such as:

- Mr
- Miss
- Mrs
- Master
- Rare titles

Titles can provide information about gender, age and social status.

Rare titles are grouped together to prevent the model from creating
too many sparse categories.


In [ ]:
# ============================================================
# TITLE EXTRACTION
# ============================================================

df["Title"] = (
    df["Name"]
    .str.extract(
        r",\s*([^.]*)\.",
        expand=False
    )
    .str.strip()
)

print("Original titles:")
display(
    df["Title"]
    .value_counts()
    .to_frame("Count")
)

common_titles = [
    "Mr",
    "Miss",
    "Mrs",
    "Master"
]

df["Title"] = df["Title"].where(
    df["Title"].isin(common_titles),
    "Rare"
)

print("\nGrouped titles:")
display(
    df["Title"]
    .value_counts()
    .to_frame("Count")
)


## 6. Extract Cabin Deck

Cabin values often contain a letter identifying the deck.

For example:

`C85 → C`

Missing cabin values are assigned the category `Unknown`.

We also create a separate binary variable indicating whether
cabin information is available.


In [ ]:
# ============================================================
# CABIN FEATURES
# ============================================================

df["Deck"] = (
    df["Cabin"]
    .fillna("Unknown")
    .astype(str)
    .str[0]
)

df["CabinKnown"] = (
    df["Cabin"].notna()
).astype(int)

display(
    df[
        [
            "Cabin",
            "Deck",
            "CabinKnown"
        ]
    ].head(15)
)

print("\nDeck distribution:")
display(
    df["Deck"]
    .value_counts()
    .to_frame("Passengers")
)


## 7. Ticket Group Size

Passengers sharing a ticket number may have been traveling
as part of the same group.

We calculate the number of passengers associated with each ticket.


In [ ]:
# ============================================================
# TICKET GROUP SIZE
# ============================================================

ticket_counts = df["Ticket"].value_counts()

df["TicketGroupSize"] = (
    df["Ticket"].map(ticket_counts)
)

display(
    df[
        [
            "Ticket",
            "TicketGroupSize"
        ]
    ].head(15)
)


## 8. Fare Per Person

Some ticket numbers represent groups of passengers.

The total fare can therefore represent a group rather than
one individual.

We calculate:

FarePerPerson = Fare / TicketGroupSize


In [ ]:
# ============================================================
# FARE PER PERSON
# ============================================================

df["FarePerPerson"] = (
    df["Fare"] /
    df["TicketGroupSize"].replace(0, 1)
)

display(
    df[
        [
            "Fare",
            "TicketGroupSize",
            "FarePerPerson"
        ]
    ].head(15)
)


## 9. Ticket Prefix

Ticket numbers sometimes contain letters or other non-numeric
characters.

We extract the non-numeric prefix to create a categorical
ticket-type feature.


In [ ]:
# ============================================================
# TICKET PREFIX
# ============================================================

df["TicketPrefix"] = (
    df["Ticket"]
    .astype(str)
    .str.replace(
        r"\d",
        "",
        regex=True
    )
    .str.replace(
        r"[\s./]+",
        "",
        regex=True
    )
)

df["TicketPrefix"] = (
    df["TicketPrefix"]
    .replace("", "NONE")
)

display(
    df[
        [
            "Ticket",
            "TicketPrefix"
        ]
    ].head(20)
)


## 10. Family Categories

Family size can also be converted into broader categories.

- Small family: 2–4 members
- Large family: 5 or more members

These binary features can help classification models identify
different family-size patterns.


In [ ]:
# ============================================================
# FAMILY CATEGORIES
# ============================================================

df["SmallFamily"] = (
    df["FamilySize"].between(2, 4)
).astype(int)

df["LargeFamily"] = (
    df["FamilySize"] >= 5
).astype(int)

display(
    df[
        [
            "FamilySize",
            "SmallFamily",
            "LargeFamily"
        ]
    ].head(15)
)


## 11. Mother Indicator

A simple analytical indicator is created for adult female
passengers who traveled with at least one child.

This feature should be interpreted as an engineered proxy rather
than a definitive identification of motherhood.


In [ ]:
# ============================================================
# MOTHER INDICATOR
# ============================================================

df["IsMother"] = (
    (df["Sex"] == "female") &
    (df["Parch"] > 0) &
    (df["Age"] > 18)
).astype(int)

display(
    df[
        [
            "Sex",
            "Age",
            "Parch",
            "IsMother"
        ]
    ].head(20)
)


## 12. Review All Engineered Features


In [ ]:
# ============================================================
# ENGINEERED FEATURE REVIEW
# ============================================================

engineered_features = [
    "FamilySize",
    "IsAlone",
    "AgeGroup",
    "Title",
    "Deck",
    "CabinKnown",
    "TicketGroupSize",
    "FarePerPerson",
    "TicketPrefix",
    "SmallFamily",
    "LargeFamily",
    "IsMother"
]

print("Engineered features:\n")

for feature in engineered_features:
    print(f"✓ {feature}")


In [ ]:
# ============================================================
# ENGINEERED DATASET PREVIEW
# ============================================================

display(
    df[
        [
            "PassengerId",
            "Survived",
            "Pclass",
            "Sex",
            "Age",
            "FamilySize",
            "IsAlone",
            "AgeGroup",
            "Title",
            "Deck",
            "CabinKnown",
            "TicketGroupSize",
            "FarePerPerson",
            "TicketPrefix",
            "SmallFamily",
            "LargeFamily",
            "IsMother"
        ]
    ].head(20)
)


## 13. Missing Values After Feature Engineering

Feature engineering should not accidentally create unexpected
missing values.

We therefore check the resulting dataset.


In [ ]:
# ============================================================
# MISSING VALUE CHECK
# ============================================================

missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (
        df.isnull().mean() * 100
    )
})

missing_summary = (
    missing_summary
    .sort_values(
        "Missing Percentage",
        ascending=False
    )
)

display(missing_summary)


## 14. Final ML Dataset

The following raw columns are not directly used by the model:

- PassengerId
- Name
- Ticket
- Cabin

The information from Name, Ticket and Cabin has already been
converted into engineered variables.

The target `Survived` is kept separately.


In [ ]:
# ============================================================
# FINAL ML DATASET
# ============================================================

TARGET = "Survived"

columns_to_drop = [
    TARGET,
    "PassengerId",
    "Name",
    "Ticket",
    "Cabin"
]

X = df.drop(
    columns=columns_to_drop,
    errors="ignore"
)

y = df[TARGET].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFinal features:")
print(X.columns.tolist())


In [ ]:
# ============================================================
# SAVE PROCESSED DATA
# ============================================================

PROCESSED_DIR = (
    PROJECT_ROOT /
    "data" /
    "processed"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

processed_path = (
    PROCESSED_DIR /
    "titanic_engineered.csv"
)

df.to_csv(
    processed_path,
    index=False
)

print(
    f"Saved engineered dataset:\n{processed_path}"
)


In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================

print("=" * 70)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 70)

print(f"Original shape: {pd.read_csv(TRAIN_PATH).shape}")
print(f"Engineered shape: {df.shape}")
print(f"ML feature count: {X.shape[1]}")
print(f"Saved file: {processed_path}")
